# Multi-Asset CTA Strategy V2 — Transition Strategy

## 05 — HMM Latent-State Transition Modelling

This version completes the HMM research programme rather than stopping at a preprocessing diagnostic.

It preserves the core safeguards:
- continuous weekly market-time state estimation;
- separate market sequences;
- causal forward filtering only;
- expanding chronological OOS evaluation;
- frozen Book 04 `RF_unweighted_C_plus_SC_no_MACD` probabilities;
- no genuine/failed labels used to fit transformations, PCA or latent states.

The decisive question is whether any transformed latent-state representation adds OOS information beyond frozen Book 04.


## Research design

Book 05 now separates two experimental axes.

### A. Information set / HMM architecture

1. **PRICE** — directional and price-derived state only.
2. **VOLATILITY** — volatility state only, deliberately independent of price direction.
3. **JOINT** — price and volatility variables in one HMM.
4. **ECONOMIC_COMPACT** — a deliberately small, economically designed set of relatively non-redundant state variables.
5. **PARALLEL** — fit PRICE and VOLATILITY HMMs separately and combine only their causal state outputs downstream. This tests whether forcing direction and volatility into one latent state is harmful.

### B. Preprocessing candidates

The tournament compares:
- `RAW` — finite, economically scaled factors with only mandatory sanitation;
- `STATIONARY` — stationary-by-construction transforms/relative measures;
- `STANDARDIZED` — training-fold z-scores;
- `WINSORIZED_STANDARDIZED` — training-fold tail clipping then z-scoring;
- `NONLINEAR_STANDARDIZED` — economically appropriate nonlinear transforms (e.g. log positive volatility; signed-log for selected heavy-tailed signed variables) then scaling;
- `PCA` — stationary + standardized inputs followed by PCA;
- `ROBUST_PCA` — stationary + winsorized + standardized inputs followed by PCA;
- `WHITENED_PCA` — robust PCA with whitening.

The intended transformation order is:

\[
\text{stationarize} \rightarrow \text{winsorize/transform} \rightarrow
\text{standardize} \rightarrow \text{PCA}
\]

All fitted preprocessing parameters — clipping thresholds, means, standard deviations, PCA loadings and retained components — are estimated **inside the pre-test training sample only**.

### Staged tournament

To avoid a combinatorial specification-mining exercise:

**Stage A — preprocessing screen:** PRICE, VOLATILITY and JOINT models are compared under a fixed compact state-count screen.

**Stage B — architecture screen:** the strongest preprocessing candidates are carried into PRICE, VOLATILITY, JOINT, ECONOMIC_COMPACT and PARALLEL architectures.

**Stage C — state-complexity robustness:** finalists alone receive the wider \(K=3,\ldots,10\) search.

**Stage D — frozen incremental test:** the final HMM representation is locked and tested against the exact frozen Book 04 OOS probability.

The production hurdle remains:

\[
\boxed{\text{Frozen Book 04 + HMM} > \text{Frozen Book 04}}
\]

A higher HMM likelihood or a better standalone HMM AUC is insufficient by itself.


In [ ]:
# =========================================================
# 1) INSTALLS, IMPORTS, DRIVE, PATHS
# =========================================================
!pip -q install pyarrow scikit-learn hmmlearn

from pathlib import Path
import json
import warnings
import math
import numpy as np
import pandas as pd

from google.colab import drive
drive.mount("/content/drive")

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
)

from hmmlearn.hmm import GaussianHMM

warnings.filterwarnings("ignore")

PROJECT = Path("/content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2")
V201 = PROJECT / "v2.01"
V203 = PROJECT / "v2.03"
V204 = PROJECT / "v2.04"
V205 = PROJECT / "v2.05"

# Canonical Book 05 output directories. Defined before any downstream writes.
CONFIG_DIR = V205 / "config"
DATA_DIR = V205 / "data"
RESULTS_DIR = V205 / "results"

for p in [CONFIG_DIR, DATA_DIR, RESULTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

PATHS = {
    "signal_prices": V201 / "data" / "processed" / "v2_01_signal_prices.parquet",
    "candidate_features_parquet": V203 / "data" / "v2_03_candidate_features.parquet",
    "candidate_features_csv": V203 / "results" / "v2_03_candidate_features.csv",
    "book04_predictions": V204 / "data" / "v2_04_unweighted_predictions.parquet",
}

CONFIG = {
    "book": "V2.05",
    "architecture": "continuous_weekly_hmm",
    "research_end": "2025-12-31",
    "first_test_year": 2008,

    "weekly_rule": "W-FRI",
    "minimum_market_weeks": 260,

    "state_grid": [3, 4, 5, 6],
    "validation_years": 2,
    "state_selection_anchor_years": [2008, 2012, 2016, 2020, 2024],
    "covariance_type": "diag",
    "hmm_n_iter": 100,
    "hmm_tol": 1e-3,
    "hmm_random_state": 42,

    "migration_lookbacks": [1, 2, 4, 8],

    "rf_estimators": 500,
    "rf_min_samples_leaf": 20,
    "rf_max_features": "sqrt",
    "random_state": 42,

    "top_quantile": 0.20,

    # Definitive transformed-HMM tournament
    "tournament_k": 8,
    "tournament_hmm_n_iter": 75,
    "tournament_hmm_tol": 1e-3,
    "tournament_first_test_year": 2008,
    "run_extended_k_robustness": False,
}

print("Project:", PROJECT)
print("Book 05 output:", V205)
print(json.dumps(CONFIG, indent=2))


# =========================================================
# BOOK 05 DEFINITIVE TOURNAMENT CONTROLS
# =========================================================
PREPROCESSING_METHODS = [
    "RAW",
    "STATIONARY",
    "STANDARDIZED",
    "WINSORIZED_STANDARDIZED",
    "NONLINEAR_STANDARDIZED",
    "PCA",
    "ROBUST_PCA",
    "WHITENED_PCA",
]

HMM_ARCHITECTURES = [
    "PRICE",
    "VOLATILITY",
    "JOINT",
    "ECONOMIC_COMPACT",
    "PARALLEL",
]

# Staged search: do NOT exhaustively optimize every combination.
STAGE_A_K_GRID = [4, 6]
STAGE_A_METHODS = PREPROCESSING_METHODS
STAGE_A_ARCHITECTURES = ["PRICE", "VOLATILITY", "JOINT"]

STAGE_B_TOP_PREPROCESSORS = 2
STAGE_C_K_GRID = list(range(3, 11))
PCA_COMPONENT_GRID = [3, 5, 7, 10]
PCA_VARIANCE_REPORT_LEVELS = [0.80, 0.90, 0.95]

WINSOR_LOWER = 0.01
WINSOR_UPPER = 0.99

# Selection is unsupervised: validation log-likelihood per observation.
# Transition labels are reserved for OOS evaluation only.
TOURNAMENT_SELECTION_METRIC = "validation_loglik_per_obs"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.0/166.0 kB 1.3 MB/s eta 0:00:00
Mounted at /content/drive
Project: /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2
Book 05 output: /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2/v2.05
{
  "book": "V2.05",
  "architecture": "continuous_weekly_hmm",
  "research_end": "2025-12-31",
  "first_test_year": 2008,
  "weekly_rule": "W-FRI",
  "minimum_market_weeks": 260,
  "state_grid": [
    3,
    4,
    5,
    6
  ],
  "validation_years": 2,
  "state_selection_anchor_years": [
    2008,
    2012,
    2016,
    2020,
    2024
  ],
  "covariance_type": "diag",
  "hmm_n_iter": 100,
  "hmm_tol": 0.001,
  "hmm_random_state": 42,
  "migration_lookbacks": [
    1,
    2,
    4,
    8
  ],
  "rf_estimators": 500,
  "rf_min_samples_leaf": 20,
  "rf_max_features": "sqrt",
  "random_state": 42,
  "top_quantile": 0.2,
  "tournament_k": 8,
  "tournament_hmm_n_iter": 75,
  "tournament_hmm_tol"

## Feature families and preprocessing engine

The definitions below deliberately distinguish **price/trend state** from **volatility state**. The stationary representation is used as the default foundation for PCA variants.

Absolute price levels are never used as HMM emissions. Variables already expressed as returns, normalized spreads, bounded drawdowns or relative volatility measures are treated as stationary-by-construction approximations. Positive volatility levels receive log transforms in the nonlinear specification.

PCA component count is chosen **without genuine/failed labels**. For each training fold the notebook reports explained variance and evaluates the pre-specified component grid using chronological HMM validation likelihood.

In [ ]:
# =========================================================
# FEATURE FAMILIES + PREPROCESSING ENGINE
# =========================================================
from sklearn.base import clone
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.impute import SimpleImputer

PRICE_FEATURES = [
    "ret_1w",
    "mom_4w",
    "mom_13w",
    "mom_26w",
    "mom_52w",
    "ema_13_52_spread",
    "drawdown_52w",
    "recovery_52w",
    "sc_osc",
    "sc_signal",
    "sc_spread",
    "sc_osc_slope_4w",
    "sc_signal_slope_4w",
]

VOLATILITY_FEATURES = [
    "rv_4w",
    "rv_13w",
]

# Compact set intentionally limits redundant views of the same phenomenon.
ECONOMIC_COMPACT_FEATURES = [
    "mom_13w",
    "mom_52w",
    "ema_13_52_spread",
    "rv_13w",
    "drawdown_52w",
    "sc_spread",
]

def add_stationary_relative_features(df):
    """Create additional stationary/relative observables using past/current data only."""
    z = df.copy().sort_values(["market", "week"])
    g = z.groupby("market", group_keys=False)

    if {"rv_4w", "rv_13w"}.issubset(z.columns):
        z["log_rv_4w"] = np.log(z["rv_4w"].clip(lower=1e-12))
        z["log_rv_13w"] = np.log(z["rv_13w"].clip(lower=1e-12))
        z["log_vol_ratio_4_13"] = np.log(
            z["rv_4w"].clip(lower=1e-12) / z["rv_13w"].clip(lower=1e-12)
        )
        z["d_log_rv_4w_4w"] = g["log_rv_4w"].diff(4)
        z["d_log_rv_13w_4w"] = g["log_rv_13w"].diff(4)

    if "ret_1w" in z.columns:
        # Vol-of-vol / return dispersion proxy from weekly returns.
        z["rv_of_rv_13w"] = g["ret_1w"].transform(
            lambda s: s.rolling(13, min_periods=8).std()
        )

    return z

STATIONARY_PRICE_FEATURES = PRICE_FEATURES.copy()
STATIONARY_VOL_FEATURES = [
    "log_rv_4w",
    "log_rv_13w",
    "log_vol_ratio_4_13",
    "d_log_rv_4w_4w",
    "d_log_rv_13w_4w",
    "rv_of_rv_13w",
]

def signed_log1p(x):
    x = np.asarray(x, dtype=float)
    return np.sign(x) * np.log1p(np.abs(x))

class FoldPreprocessor:
    """Training-fold-only preprocessing; no test information enters fitted transforms."""
    def __init__(self, method, n_components=None):
        self.method = method
        self.n_components = n_components
        self.columns_ = None
        self.imputer_ = SimpleImputer(strategy="median")
        self.scaler_ = None
        self.pca_ = None
        self.lower_ = None
        self.upper_ = None

    def _winsor_fit(self, X):
        self.lower_ = np.nanquantile(X, WINSOR_LOWER, axis=0)
        self.upper_ = np.nanquantile(X, WINSOR_UPPER, axis=0)

    def _winsor_apply(self, X):
        return np.clip(X, self.lower_, self.upper_)

    def fit(self, df, columns):
        self.columns_ = list(columns)
        X = df[self.columns_].replace([np.inf, -np.inf], np.nan).to_numpy(float)
        X = self.imputer_.fit_transform(X)

        if self.method in {"WINSORIZED_STANDARDIZED", "ROBUST_PCA", "WHITENED_PCA"}:
            self._winsor_fit(X)
            X = self._winsor_apply(X)

        if self.method == "NONLINEAR_STANDARDIZED":
            X = signed_log1p(X)

        if self.method == "RAW":
            self.scaler_ = None
        else:
            self.scaler_ = StandardScaler().fit(X)
            X = self.scaler_.transform(X)

        if self.method in {"PCA", "ROBUST_PCA", "WHITENED_PCA"}:
            max_pc = min(X.shape[1], X.shape[0] - 1)
            n_pc = min(self.n_components or max_pc, max_pc)
            self.pca_ = PCA(
                n_components=n_pc,
                whiten=(self.method == "WHITENED_PCA"),
                random_state=CONFIG.get("random_state", 42),
            ).fit(X)
        return self

    def transform(self, df):
        X = df[self.columns_].replace([np.inf, -np.inf], np.nan).to_numpy(float)
        X = self.imputer_.transform(X)

        if self.method in {"WINSORIZED_STANDARDIZED", "ROBUST_PCA", "WHITENED_PCA"}:
            X = self._winsor_apply(X)

        if self.method == "NONLINEAR_STANDARDIZED":
            X = signed_log1p(X)

        if self.scaler_ is not None:
            X = self.scaler_.transform(X)

        if self.pca_ is not None:
            X = self.pca_.transform(X)
        return X

    @property
    def explained_variance_ratio_(self):
        return None if self.pca_ is None else self.pca_.explained_variance_ratio_

    @property
    def actual_n_components_(self):
        return None if self.pca_ is None else int(self.pca_.n_components_)

def architecture_columns(architecture, method):
    stationary = method in {
        "STATIONARY", "PCA", "ROBUST_PCA", "WHITENED_PCA"
    }

    if architecture == "PRICE":
        return STATIONARY_PRICE_FEATURES if stationary else PRICE_FEATURES
    if architecture == "VOLATILITY":
        return STATIONARY_VOL_FEATURES if stationary else VOLATILITY_FEATURES
    if architecture == "JOINT":
        p = STATIONARY_PRICE_FEATURES if stationary else PRICE_FEATURES
        v = STATIONARY_VOL_FEATURES if stationary else VOLATILITY_FEATURES
        return p + v
    if architecture == "ECONOMIC_COMPACT":
        return ECONOMIC_COMPACT_FEATURES
    raise ValueError(f"Unsupported direct architecture: {architecture}")


In [ ]:
# =========================================================
# 2) LOAD BOOK 03 CANDIDATES + EXACT BOOK 04 OOS BENCHMARK
# =========================================================
if PATHS["candidate_features_parquet"].exists():
    candidates = pd.read_parquet(PATHS["candidate_features_parquet"])
    candidate_source = PATHS["candidate_features_parquet"]
elif PATHS["candidate_features_csv"].exists():
    candidates = pd.read_csv(PATHS["candidate_features_csv"])
    candidate_source = PATHS["candidate_features_csv"]
else:
    raise FileNotFoundError(
        "Book 03 candidate feature file not found.\n"
        f"Tried:\n{PATHS['candidate_features_parquet']}\n"
        f"{PATHS['candidate_features_csv']}"
    )

candidates["candidate_date"] = pd.to_datetime(candidates["candidate_date"])
candidates["target"] = (candidates["label"] == "genuine").astype(int)
candidates["test_year"] = candidates["candidate_date"].dt.year

if not PATHS["book04_predictions"].exists():
    raise FileNotFoundError(
        "Exact Book 04 OOS prediction file not found:\n"
        f"{PATHS['book04_predictions']}\n\n"
        "Book 05 deliberately requires the saved Book 04 predictions so that "
        "the benchmark is truly frozen rather than approximately reconstructed."
    )

b4 = pd.read_parquet(PATHS["book04_predictions"])
b4["candidate_date"] = pd.to_datetime(b4["candidate_date"])

print("Book 03 source:", candidate_source)
print("Book 03 candidates:", len(candidates))
print("Book 04 prediction rows:", len(b4))
print("Book 04 models:", sorted(b4["model"].dropna().unique().tolist()))

FROZEN_MODEL = "RF_unweighted_C_plus_SC_no_MACD"
if FROZEN_MODEL not in set(b4["model"].astype(str)):
    raise ValueError(
        f"Frozen model '{FROZEN_MODEL}' not found in Book 04 predictions.\n"
        f"Available models: {sorted(b4['model'].astype(str).unique().tolist())}"
    )

frozen = b4[b4["model"].astype(str) == FROZEN_MODEL].copy()

prob_candidates = [
    "prob_genuine",
    "predicted_probability",
    "probability",
    "prediction",
]
prob_col = next((c for c in prob_candidates if c in frozen.columns), None)
if prob_col is None:
    raise ValueError(
        "Could not find Book 04 probability column. "
        f"Available columns: {list(frozen.columns)}"
    )

frozen = frozen.rename(columns={prob_col: "book04_prob_genuine"})

merge_keys = ["candidate_date", "market"]
for extra in ["candidate_direction", "category"]:
    if extra in candidates.columns and extra in frozen.columns:
        merge_keys.append(extra)

frozen_cols = merge_keys + ["book04_prob_genuine"]
if "target" in frozen.columns:
    frozen_cols.append("target")

base = candidates.merge(
    frozen[frozen_cols].drop_duplicates(merge_keys),
    on=merge_keys,
    how="inner",
    suffixes=("", "_b4"),
    validate="one_to_one",
)

print("Exact frozen Book 04 events merged:", len(base))
print("Frozen date range:", base["candidate_date"].min(), "to", base["candidate_date"].max())
print("Frozen genuine rate:", round(base["target"].mean(), 6))
print("Frozen mean predicted probability:", round(base["book04_prob_genuine"].mean(), 6))


Book 03 source: /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2/v2.03/data/v2_03_candidate_features.parquet
Book 03 candidates: 3462
Book 04 prediction rows: 8061
Book 04 models: ['LOGIT_unweighted_full', 'RF_unweighted_C_plus_SC_no_MACD', 'RF_unweighted_full']
Exact frozen Book 04 events merged: 2687
Frozen date range: 2008-01-03 00:00:00 to 2025-12-31 00:00:00
Frozen genuine rate: 0.32527
Frozen mean predicted probability: 0.324119


In [ ]:
# =========================================================
# 3) LOAD + NORMALISE BOOK 01 SIGNAL PRICE PANEL
# =========================================================
if not PATHS["signal_prices"].exists():
    raise FileNotFoundError(
        "Book 01 signal-price panel not found:\n"
        f"{PATHS['signal_prices']}"
    )

raw_prices = pd.read_parquet(PATHS["signal_prices"])

def normalise_price_panel(df):
    '''
    Return wide dataframe:
        index = datetime
        columns = market names
        values = signal prices
    Supports common wide and long layouts.
    '''
    x = df.copy()

    lower = {str(c).lower(): c for c in x.columns}
    date_col = next((lower[k] for k in ["date", "datetime", "timestamp"] if k in lower), None)
    market_col = next((lower[k] for k in ["market", "asset", "name"] if k in lower), None)
    price_col = next((lower[k] for k in ["price", "close", "signal_price", "value"] if k in lower), None)

    if date_col is not None and market_col is not None and price_col is not None:
        x[date_col] = pd.to_datetime(x[date_col])
        wide = x.pivot_table(
            index=date_col,
            columns=market_col,
            values=price_col,
            aggfunc="last",
        )
        return wide.sort_index()

    if date_col is not None:
        x[date_col] = pd.to_datetime(x[date_col])
        x = x.set_index(date_col)

    if not isinstance(x.index, pd.DatetimeIndex):
        try:
            x.index = pd.to_datetime(x.index)
        except Exception as e:
            raise ValueError(
                "Could not interpret Book 01 price panel as long or wide time series."
            ) from e

    x = x.apply(pd.to_numeric, errors="coerce")
    return x.sort_index()

prices = normalise_price_panel(raw_prices)
prices = prices.loc[:pd.Timestamp(CONFIG["research_end"])]

print("Daily signal panel shape:", prices.shape)
print("Date range:", prices.index.min(), "to", prices.index.max())
print("Example columns:", list(prices.columns[:10]))


Daily signal panel shape: (10571, 53)
Date range: 1990-01-01 00:00:00 to 2025-12-31 00:00:00
Example columns: ['20+ Year Treasury ETF', '30-Day Fed Funds', 'US 10Y Treasury Note', 'US 2Y Treasury Note', 'US 30Y Treasury Bond', 'US 5Y Treasury Note', 'US Ultra Treasury Bond', 'Aluminium', 'Brent Crude Oil', 'Cocoa']


In [ ]:
# =========================================================
# 4) ROBUST MARKET-NAME MATCHING
# =========================================================
def canon(s):
    return (
        str(s).strip().lower()
        .replace("&", "and")
        .replace("/", "")
        .replace("-", "")
        .replace("_", "")
        .replace(" ", "")
        .replace(".", "")
        .replace("^", "")
    )

price_lookup = {canon(c): c for c in prices.columns}

market_map = {}
unmatched = []

for m in sorted(base["market"].astype(str).unique()):
    if m in prices.columns:
        market_map[m] = m
        continue

    cm = canon(m)
    if cm in price_lookup:
        market_map[m] = price_lookup[cm]
        continue

    hits = [
        c for c in prices.columns
        if cm in canon(c) or canon(c) in cm
    ]
    if len(hits) == 1:
        market_map[m] = hits[0]
    else:
        unmatched.append((m, hits[:5]))

print("Matched candidate markets:", len(market_map))
print("Unmatched:", len(unmatched))
if unmatched:
    print("\nUnmatched examples:")
    for row in unmatched[:20]:
        print(" ", row)

if len(market_map) < max(1, int(0.80 * base["market"].nunique())):
    raise ValueError(
        "Fewer than 80% of candidate markets matched Book 01 signal-price columns. "
        "Inspect the printed unmatched names before proceeding."
    )


Matched candidate markets: 53
Unmatched: 0


## Continuous weekly HMM observables

The HMM is intentionally fitted to a compact **generic market-state vector**, not candidate labels and not candidate-direction transforms.

Weekly observables:

- 1-week return;
- 4-, 13-, 26-, and 52-week momentum;
- 13/52 EMA spread;
- 4- and 13-week realized volatility;
- 52-week drawdown and recovery;
- SuperbCommand-style weekly oscillator `(EMA20 - EMA50) / price`;
- SuperbCommand signal line;
- oscillator-minus-signal spread;
- 4-week oscillator and signal slopes.

The adaptive SuperTrend portion is deliberately not reconstructed from a close-only panel. The HMM should first prove that **continuous latent-state dynamics themselves** add value.


In [ ]:
# =========================================================
# 5) BUILD CONTINUOUS WEEKLY FEATURE PANEL
# =========================================================
def ema(s, span):
    return s.ewm(span=span, adjust=False, min_periods=span).mean()

def build_weekly_features(price_series, market_name):
    s = price_series.dropna().astype(float)
    if len(s) == 0:
        return pd.DataFrame()

    w = s.resample(CONFIG["weekly_rule"]).last().dropna()
    if len(w) < CONFIG["minimum_market_weeks"]:
        return pd.DataFrame()

    r1 = w.pct_change()

    fast_sc = ema(w, 20)
    slow_sc = ema(w, 50)
    osc = (fast_sc - slow_sc) / w
    sig = ema(osc, 25)
    spread = osc - sig

    e13 = ema(w, 13)
    e52 = ema(w, 52)

    rolling_high_52 = w.rolling(52, min_periods=52).max()
    rolling_low_52 = w.rolling(52, min_periods=52).min()

    f = pd.DataFrame(index=w.index)
    f["price"] = w
    f["ret_1w"] = r1
    f["mom_4w"] = w.pct_change(4)
    f["mom_13w"] = w.pct_change(13)
    f["mom_26w"] = w.pct_change(26)
    f["mom_52w"] = w.pct_change(52)
    f["ema_13_52_spread"] = (e13 - e52) / w
    f["rv_4w"] = r1.rolling(4, min_periods=4).std()
    f["rv_13w"] = r1.rolling(13, min_periods=13).std()
    f["drawdown_52w"] = w / rolling_high_52 - 1.0
    f["recovery_52w"] = w / rolling_low_52 - 1.0
    f["sc_osc"] = osc
    f["sc_signal"] = sig
    f["sc_spread"] = spread
    f["sc_osc_slope_4w"] = osc - osc.shift(4)
    f["sc_signal_slope_4w"] = sig - sig.shift(4)
    f["market"] = market_name
    f["week"] = f.index

    return f.reset_index(drop=True)

weekly_parts = []

for candidate_market, price_col in market_map.items():
    f = build_weekly_features(prices[price_col], candidate_market)
    if len(f):
        weekly_parts.append(f)

weekly = pd.concat(weekly_parts, ignore_index=True)
weekly = weekly.sort_values(["market", "week"]).reset_index(drop=True)

HMM_OBSERVABLES = [
    "ret_1w",
    "mom_4w",
    "mom_13w",
    "mom_26w",
    "mom_52w",
    "ema_13_52_spread",
    "rv_4w",
    "rv_13w",
    "drawdown_52w",
    "recovery_52w",
    "sc_osc",
    "sc_signal",
    "sc_spread",
    "sc_osc_slope_4w",
    "sc_signal_slope_4w",
]

weekly = weekly.dropna(subset=HMM_OBSERVABLES).copy()

print("Continuous weekly rows:", len(weekly))
print("Markets:", weekly["market"].nunique())
print("Weekly date range:", weekly["week"].min(), "to", weekly["week"].max())
print("HMM observables:", len(HMM_OBSERVABLES))
print()
print(weekly.groupby("market").size().describe())


Continuous weekly rows: 68957
Markets: 53
Weekly date range: 1991-06-28 00:00:00 to 2026-01-02 00:00:00
HMM observables: 15

count      53.000000
mean     1301.075472
std       378.459522
min       203.000000
25%      1118.000000
50%      1247.000000
75%      1651.000000
max      1802.000000
dtype: float64


## Causal HMM filtering

`hmmlearn.predict_proba()` uses forward-backward smoothing and can therefore allow later observations in the same sequence to influence the posterior state probability at an earlier date.

That is inappropriate for a trading signal.

This notebook instead implements the HMM **forward filter directly**:

\[
\alpha_t(j)
\propto
p(x_t \mid S_t=j)
\sum_i \alpha_{t-1}(i)P_{ij}
\]

Therefore every state-probability vector at week \(t\) uses information available only through week \(t\).


In [ ]:
# =========================================================
# 6) HMM HELPERS: SEQUENCES, EMISSIONS, CAUSAL FILTER
# =========================================================
def gaussian_diag_logpdf(X, means, covars):
    X = np.asarray(X, float)
    means = np.asarray(means, float)
    covars = np.asarray(covars, float)

    if covars.ndim == 3:
        covars = np.array([np.diag(c) for c in covars])

    covars = np.clip(covars, 1e-8, None)

    diff = X[:, None, :] - means[None, :, :]
    log_det = np.log(covars).sum(axis=1)
    quad = (diff * diff / covars[None, :, :]).sum(axis=2)
    d = X.shape[1]

    return -0.5 * (d * np.log(2 * np.pi) + log_det[None, :] + quad)

def logsumexp(a, axis=None):
    a = np.asarray(a)
    m = np.max(a, axis=axis, keepdims=True)
    z = m + np.log(np.sum(np.exp(a - m), axis=axis, keepdims=True))
    if axis is not None:
        z = np.squeeze(z, axis=axis)
    return z

def causal_filter(model, X):
    log_emission = gaussian_diag_logpdf(X, model.means_, model.covars_)
    T, K = log_emission.shape

    log_start = np.log(np.clip(model.startprob_, 1e-15, 1.0))
    log_trans = np.log(np.clip(model.transmat_, 1e-15, 1.0))

    out = np.zeros((T, K), float)

    log_alpha = log_start + log_emission[0]
    log_alpha -= logsumexp(log_alpha)
    out[0] = np.exp(log_alpha)

    for t in range(1, T):
        pred = logsumexp(
            log_alpha[:, None] + log_trans,
            axis=0,
        )
        log_alpha = pred + log_emission[t]
        log_alpha -= logsumexp(log_alpha)
        out[t] = np.exp(log_alpha)

    return out

def prepare_sequences(df, cols, imputer=None, scaler=None, fit=False):
    x = df.copy().sort_values(["market", "week"])

    if fit:
        imputer = SimpleImputer(strategy="median")
        scaler = StandardScaler()
        Z = imputer.fit_transform(x[cols])
        Z = scaler.fit_transform(Z)
    else:
        Z = imputer.transform(x[cols])
        Z = scaler.transform(Z)

    lengths = x.groupby("market", sort=False).size().tolist()
    return x, Z, lengths, imputer, scaler

def fit_hmm(Z, lengths, n_states):
    model = GaussianHMM(
        n_components=n_states,
        covariance_type=CONFIG["covariance_type"],
        n_iter=CONFIG["hmm_n_iter"],
        tol=CONFIG["hmm_tol"],
        random_state=CONFIG["hmm_random_state"],
    )
    model.fit(Z, lengths)
    return model


## Efficient chronological state-count selection

The previous regenerated Book 05 performed a full state-count search for every OOS year. That is methodologically valid but computationally wasteful.

This optimized version preserves chronology while reducing redundant fitting:

1. State count is re-selected only on **anchor years**: 2008, 2012, 2016, 2020, and 2024.
2. Between anchor years, the most recently selected \(K\) is carried forward.
3. The search grid is reduced to `K = {3, 4, 5, 6}`.
4. HMM estimation is capped at **100 EM iterations** with tolerance `1e-3`.
5. The selected \(K\) is still refitted on **all pre-test history** for every OOS year.
6. The test year is never used to select \(K\).

This cuts the expensive model-selection fits by roughly an order of magnitude without changing the actual latent-state hypothesis.

## Staged HMM tournament

This section screens preprocessing and architecture choices **without using transition labels for model selection**.

To keep runtime practical:
- Stage A uses `K={4,6}` and the three direct information sets.
- Stage B retains only the strongest preprocessing methods by chronological validation likelihood and adds the compact and parallel architectures.
- Stage C applies `K=3..10` only to finalists.
- Stage D evaluates the locked winner against Book 04.

`PARALLEL` is intentionally handled as two separately fitted latent processes. Its downstream candidate features concatenate the causal PRICE-HMM and VOLATILITY-HMM summaries; it is never treated as a single joint emission model.
> **Runtime note:** `hmmlearn` may print `Model is not converging` for individual trial
> specifications when the final EM step slightly reduces likelihood. Stage A records convergence
> status; these are warnings, not notebook-stopping errors. Failed specifications are captured
> in the screening table rather than terminating the tournament.
\n\n**Important:** absolute Gaussian validation log-likelihood is only comparable within the same transformed representation. Scaling and dimensionality change the density/Jacobian, so Stage A must not mechanically rank RAW against standardized/PCA specifications. Cross-preprocessing success is decided by the chronological candidate-level OOS test below.

In [ ]:
# =========================================================
# PREPARE STATIONARY / RELATIVE WEEKLY PANEL
# =========================================================
# Execution-order guard for tournament outputs.
for _name in ("CONFIG_DIR", "DATA_DIR", "RESULTS_DIR"):
    if _name not in globals():
        raise RuntimeError(
            f"{_name} is not initialized. Restart the runtime and run Book 05 from the top."
        )

weekly = add_stationary_relative_features(weekly)

def contiguous_sequences(df, columns, preprocessor=None, fit_preprocessor=False,
                         method="STANDARDIZED", n_components=None):
    d = df.sort_values(["market", "week"]).copy()

    if preprocessor is None:
        preprocessor = FoldPreprocessor(method, n_components=n_components)
        preprocessor.fit(d, columns)

    X = preprocessor.transform(d)

    lengths = d.groupby("market", sort=False).size().astype(int).tolist()
    return d, X, lengths, preprocessor

def fit_hmm_matrix(X, lengths, n_states):
    model = GaussianHMM(
        n_components=n_states,
        covariance_type="diag",
        n_iter=CONFIG.get("hmm_n_iter", 100),
        tol=CONFIG.get("hmm_tol", 1e-3),
        random_state=CONFIG.get("random_state", 42),
    )
    model.fit(X, lengths)
    return model


def score_separate_sequences(model, Z, lengths):
    """Average HMM log-likelihood per observation, respecting market boundaries."""
    scores = []
    pos = 0
    for L in lengths:
        seq = Z[pos:pos + L]
        pos += L
        if L >= 2:
            try:
                scores.append((model.score(seq), L))
            except Exception:
                pass

    if not scores:
        return np.nan

    total_ll = sum(s for s, _ in scores)
    total_n = sum(n for _, n in scores)
    return total_ll / total_n

def chronological_unsupervised_score(train_weekly, architecture, method, k,
                                     n_components=None, validation_years=None):
    validation_years = validation_years or CONFIG["validation_years"]
    last_year = int(train_weekly["week"].dt.year.max())
    validation_start = pd.Timestamp(f"{last_year - validation_years + 1}-01-01")

    fit_df = train_weekly[train_weekly["week"] < validation_start].copy()
    val_df = train_weekly[train_weekly["week"] >= validation_start].copy()

    cols = architecture_columns(architecture, method)
    cols = [c for c in cols if c in fit_df.columns]

    _, X_fit, L_fit, prep = contiguous_sequences(
        fit_df, cols, method=method, n_components=n_components
    )
    model = fit_hmm_matrix(X_fit, L_fit, k)

    # Validation is scored as separate market sequences using training-fitted transforms.
    val_sorted = val_df.sort_values(["market", "week"]).copy()
    X_val = prep.transform(val_sorted)
    L_val = val_sorted.groupby("market", sort=False).size().astype(int).tolist()

    ll_per_obs = score_separate_sequences(model, X_val, L_val)

    ev = prep.explained_variance_ratio_
    return {
        "architecture": architecture,
        "preprocessing": method,
        "n_states": k,
        "requested_n_components": n_components,
        "actual_n_components": prep.actual_n_components_,
        "validation_loglik_per_obs": ll_per_obs,
        "fit_converged": bool(model.monitor_.converged),
        "pca_explained_variance": np.nan if ev is None else float(np.sum(ev)),
        "n_fit_rows": len(fit_df),
        "n_validation_rows": len(val_df),
    }

def run_stage_a_screen(reference_train):
    """Fast unsupervised screen. No transition labels are consulted."""
    rows = []
    for arch in STAGE_A_ARCHITECTURES:
        for method in STAGE_A_METHODS:
            pc_grid = PCA_COMPONENT_GRID if method in {
                "PCA", "ROBUST_PCA", "WHITENED_PCA"
            } else [None]

            for n_pc in pc_grid:
                for k in STAGE_A_K_GRID:
                    try:
                        r = chronological_unsupervised_score(
                            reference_train, arch, method, k, n_components=n_pc
                        )
                        rows.append(r)
                    except Exception as e:
                        rows.append({
                            "architecture": arch,
                            "preprocessing": method,
                            "n_states": k,
                            "requested_n_components": n_pc,
                            "actual_n_components": np.nan,
                            "validation_loglik_per_obs": np.nan,
                            "error": str(e),
                        })
    return pd.DataFrame(rows)

# Stage A is intentionally run once on the designated pre-OOS reference history,
# rather than re-optimizing preprocessing in every OOS year.
reference_cutoff = pd.Timestamp(f"{CONFIG['first_test_year']}-01-01")
reference_train = weekly[weekly["week"] < reference_cutoff].copy()

stage_a_results = run_stage_a_screen(reference_train)
stage_a_results.to_csv(
    RESULTS_DIR / "v2_05_preprocessing_screen.csv", index=False
)

print("Stage A preprocessing screen complete.")
display(
    stage_a_results.sort_values(
        "validation_loglik_per_obs", ascending=False
    ).head(30)
)


Stage A preprocessing screen complete.


,architecture,preprocessing,n_states,requested_n_components,actual_n_components,validation_loglik_per_obs,fit_converged,pca_explained_variance,n_fit_rows,n_validation_rows
69,JOINT,RAW,6,NaN,NaN,35.175597,True,NaN,16789,4642
68,JOINT,RAW,4,NaN,NaN,33.132961,True,NaN,16789,4642
1,PRICE,RAW,6,NaN,NaN,28.874900,True,NaN,16789,4642
0,PRICE,RAW,4,NaN,NaN,27.519050,True,NaN,16789,4642
35,VOLATILITY,RAW,6,NaN,NaN,7.217883,True,NaN,16789,4642
34,VOLATILITY,RAW,4,NaN,NaN,6.843506,True,NaN,16789,4642
49,VOLATILITY,PCA,6,7.0,6.0,0.958405,True,1.000000,16789,4642
51,VOLATILITY,PCA,6,10.0,6.0,0.958405,True,1.000000,16789,4642
50,VOLATILITY,PCA,4,10.0,6.0,0.697619,True,1.000000,16789,4642
48,VOLATILITY,PCA,4,7.0,6.0,0.697619,True,1.000000,16789,4642


In [ ]:
# =========================================================
# STAGE B/C FINALIST PLAN — THEORY-DRIVEN, NOT CROSS-SCALE LL RANKING
# =========================================================
if "RESULTS_DIR" not in globals():
    raise RuntimeError("RESULTS_DIR is not initialized. Run Book 05 from the top.")

FINALIST_SPECS = [
    {"spec_id": "LEGACY_JOINT_STANDARDIZED", "architecture": "JOINT", "preprocessing": "STANDARDIZED", "n_components": None},
    {"spec_id": "PRICE_STANDARDIZED", "architecture": "PRICE", "preprocessing": "STANDARDIZED", "n_components": None},
    {"spec_id": "PRICE_ROBUST_PCA5", "architecture": "PRICE", "preprocessing": "ROBUST_PCA", "n_components": 5},
    {"spec_id": "PRICE_WHITENED_PCA5", "architecture": "PRICE", "preprocessing": "WHITENED_PCA", "n_components": 5},
    {"spec_id": "VOL_STATIONARY", "architecture": "VOLATILITY", "preprocessing": "STATIONARY", "n_components": None},
    {"spec_id": "VOL_ROBUST_PCA3", "architecture": "VOLATILITY", "preprocessing": "ROBUST_PCA", "n_components": 3},
    {"spec_id": "JOINT_ROBUST_PCA7", "architecture": "JOINT", "preprocessing": "ROBUST_PCA", "n_components": 7},
    {"spec_id": "ECONOMIC_COMPACT_WINSOR", "architecture": "ECONOMIC_COMPACT", "preprocessing": "WINSORIZED_STANDARDIZED", "n_components": None},
]

PARALLEL_SPEC = {
    "spec_id": "PARALLEL_PRICE_PCA5_VOL_PCA3",
    "price_spec": "PRICE_ROBUST_PCA5",
    "vol_spec": "VOL_ROBUST_PCA3",
}

stage_b_plan = pd.DataFrame(FINALIST_SPECS)
parallel_row = pd.DataFrame([{
    "spec_id": PARALLEL_SPEC["spec_id"],
    "architecture": "PARALLEL",
    "preprocessing": f"{PARALLEL_SPEC['price_spec']} + {PARALLEL_SPEC['vol_spec']}",
    "n_components": np.nan,
}])
stage_b_plan = pd.concat([stage_b_plan, parallel_row], ignore_index=True)
stage_b_plan.to_csv(RESULTS_DIR / "v2_05_stage_b_plan.csv", index=False)
print("Definitive transformed-HMM finalist plan:")
display(stage_b_plan)


Definitive transformed-HMM finalist plan:


,spec_id,architecture,preprocessing,n_components
0,LEGACY_JOINT_STANDARDIZED,JOINT,STANDARDIZED,NaN
1,PRICE_STANDARDIZED,PRICE,STANDARDIZED,NaN
2,PRICE_ROBUST_PCA5,PRICE,ROBUST_PCA,5.0
3,PRICE_WHITENED_PCA5,PRICE,WHITENED_PCA,5.0
4,VOL_STATIONARY,VOLATILITY,STATIONARY,NaN
5,VOL_ROBUST_PCA3,VOLATILITY,ROBUST_PCA,3.0
6,JOINT_ROBUST_PCA7,JOINT,ROBUST_PCA,7.0
7,ECONOMIC_COMPACT_WINSOR,ECONOMIC_COMPACT,WINSORIZED_STANDARDIZED,NaN
8,PARALLEL_PRICE_PCA5_VOL_PCA3,PARALLEL,PRICE_ROBUST_PCA5 + VOL_ROBUST_PCA3,NaN


## Definitive transformed-HMM OOS tournament

The preprocessing screen is diagnostic only. The cells below actually fit each finalist HMM in every annual OOS fold, causally filter the state probabilities, extract candidate-date latent-state/migration features, and compare HMM-only and Book04+HMM models against the exact frozen Book 04 probability.

For comparability across preprocessing methods, every finalist uses fixed `K=8`. Wider `K=3..10` work is optional and reserved for finalists that earn it.

In [ ]:
# =========================================================
# 8) FINALIST HMM HELPERS: TRAINING-ONLY STATE INTERPRETATION
# =========================================================
TOURNAMENT_K = int(CONFIG["tournament_k"])
TOURNAMENT_LOOKBACKS = [1, 4, 8]

def zscore_train(x):
    x = np.asarray(x, float)
    mu = np.nanmean(x)
    sd = np.nanstd(x)
    if not np.isfinite(sd) or sd < 1e-12:
        return np.nan_to_num(x - mu)
    return np.nan_to_num((x - mu) / sd)

def training_direction_anchor(df):
    cols = [c for c in ["mom_13w", "mom_52w", "ema_13_52_spread", "sc_spread"] if c in df.columns]
    if not cols:
        return np.zeros(len(df))
    return np.mean(np.column_stack([zscore_train(df[c].to_numpy(float)) for c in cols]), axis=1)

def training_vol_anchor(df):
    if "log_rv_13w" in df.columns:
        a = zscore_train(df["log_rv_13w"].to_numpy(float))
    else:
        a = zscore_train(np.log(df["rv_13w"].clip(lower=1e-12).to_numpy(float)))
    if "log_vol_ratio_4_13" in df.columns:
        b = zscore_train(df["log_vol_ratio_4_13"].to_numpy(float))
        return 0.5 * (a + b)
    return a

def state_weighted_anchor(gamma, anchor):
    gamma = np.asarray(gamma, float)
    anchor = np.asarray(anchor, float)
    denom = gamma.sum(axis=0)
    num = (gamma * anchor[:, None]).sum(axis=0)
    out = np.divide(num, denom, out=np.zeros_like(num), where=denom > 1e-12)
    return zscore_train(out)

def fit_finalist_fold(train_df, spec):
    cols = architecture_columns(spec["architecture"], spec["preprocessing"])
    cols = [c for c in cols if c in train_df.columns]
    train_sorted, X_train, lengths, prep = contiguous_sequences(
        train_df, cols, method=spec["preprocessing"], n_components=spec["n_components"]
    )
    model = GaussianHMM(
        n_components=TOURNAMENT_K,
        covariance_type="diag",
        n_iter=CONFIG["tournament_hmm_n_iter"],
        tol=CONFIG["tournament_hmm_tol"],
        random_state=CONFIG["hmm_random_state"],
    )
    model.fit(X_train, lengths)
    gamma = model.predict_proba(X_train, lengths)

    arch = spec["architecture"]
    has_direction = arch in {"PRICE", "JOINT", "ECONOMIC_COMPACT"}
    has_volatility = arch in {"VOLATILITY", "JOINT", "ECONOMIC_COMPACT"}
    direction_state_score = state_weighted_anchor(gamma, training_direction_anchor(train_sorted)) if has_direction else None
    volatility_state_score = state_weighted_anchor(gamma, training_vol_anchor(train_sorted)) if has_volatility else None

    meta = {
        "columns": cols,
        "actual_n_components": prep.actual_n_components_,
        "pca_explained_variance": np.nan if prep.explained_variance_ratio_ is None else float(np.sum(prep.explained_variance_ratio_)),
        "converged": bool(model.monitor_.converged),
        "has_direction": has_direction,
        "has_volatility": has_volatility,
        "direction_state_score": direction_state_score,
        "volatility_state_score": volatility_state_score,
    }
    return model, prep, meta

def causal_state_frame(model, prep, hist, meta, fold_year):
    h = hist.sort_values("week").copy()
    X = prep.transform(h)
    prob = causal_filter(model, X)
    out = h[["market", "week"]].copy()
    out["fold_year"] = int(fold_year)
    out["hmm_max_prob"] = prob.max(axis=1)
    clipped = np.clip(prob, 1e-15, 1.0)
    out["hmm_entropy"] = -(clipped * np.log(clipped)).sum(axis=1)
    stay = np.diag(model.transmat_)
    out["hmm_switch_probability"] = 1.0 - (prob @ stay)
    out["hmm_expected_persistence"] = prob @ stay
    if meta["has_direction"]:
        ds = meta["direction_state_score"]
        out["hmm_direction_score"] = prob @ ds
        pos = ds > 0
        neg = ds < 0
        out["hmm_prob_positive_state"] = prob[:, pos].sum(axis=1) if pos.any() else 0.0
        out["hmm_prob_negative_state"] = prob[:, neg].sum(axis=1) if neg.any() else 0.0
    if meta["has_volatility"]:
        out["hmm_volatility_score"] = prob @ meta["volatility_state_score"]
    return out

def candidate_direction_sign(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, str):
        s = x.strip().upper()
        if "BEAR" in s and "BULL" in s and s.index("BEAR") < s.index("BULL"):
            return 1.0
        if "BULL" in s and "BEAR" in s and s.index("BULL") < s.index("BEAR"):
            return -1.0
        try:
            return float(x)
        except Exception:
            return np.nan
    return float(x)


In [ ]:
# =========================================================
# 9) ANNUAL EXPANDING OOS FITS FOR ALL DIRECT FINALISTS
# =========================================================
candidate_years = sorted(int(y) for y in base["test_year"].unique() if int(y) >= CONFIG["tournament_first_test_year"])
max_lag_weeks = max(TOURNAMENT_LOOKBACKS)
state_frames_by_spec = {}
fit_audit_rows = []

for spec in FINALIST_SPECS:
    spec_id = spec["spec_id"]
    print(f"\n{'='*80}\n{spec_id}\n{'='*80}")
    spec_parts = []
    for test_year in candidate_years:
        test_start = pd.Timestamp(f"{test_year}-01-01")
        test_end = pd.Timestamp(f"{test_year}-12-31")
        retain_start = test_start - pd.Timedelta(weeks=max_lag_weeks + 2)
        train_df = weekly[weekly["week"] < test_start].copy()
        if len(train_df) == 0:
            continue
        print(f"{spec_id} | {test_year}: fit K={TOURNAMENT_K}", flush=True)
        try:
            model, prep, meta = fit_finalist_fold(train_df, spec)
        except Exception as e:
            fit_audit_rows.append({"spec_id": spec_id, "test_year": test_year, "status": "FIT_FAILED", "error": str(e)})
            print("  FIT FAILED:", e)
            continue
        fit_audit_rows.append({
            "spec_id": spec_id, "test_year": test_year, "status": "OK",
            "converged": meta["converged"], "n_states": TOURNAMENT_K,
            "actual_n_components": meta["actual_n_components"],
            "pca_explained_variance": meta["pca_explained_variance"],
            "n_train_rows": len(train_df), "n_features": len(meta["columns"]),
        })
        markets_this_year = set(base.loc[base["test_year"] == test_year, "market"].astype(str))
        for market in sorted(markets_this_year):
            hist = weekly[(weekly["market"].astype(str) == str(market)) & (weekly["week"] <= test_end)].copy()
            if len(hist) == 0 or not (hist["week"] < test_start).any():
                continue
            try:
                sf = causal_state_frame(model, prep, hist, meta, test_year)
                sf = sf[sf["week"] >= retain_start].copy()
                sf["spec_id"] = spec_id
                spec_parts.append(sf)
            except Exception as e:
                fit_audit_rows.append({"spec_id": spec_id, "test_year": test_year, "market": market, "status": "FILTER_FAILED", "error": str(e)})
    state_frames_by_spec[spec_id] = pd.concat(spec_parts, ignore_index=True) if spec_parts else pd.DataFrame()

fit_audit = pd.DataFrame(fit_audit_rows)
fit_audit.to_csv(RESULTS_DIR / "v2_05_transformed_hmm_fit_audit.csv", index=False)
print("\nDirect finalist fits complete.")
display(fit_audit.groupby(["spec_id", "status"]).size().rename("rows").reset_index())



LEGACY_JOINT_STANDARDIZED
LEGACY_JOINT_STANDARDIZED | 2008: fit K=8
LEGACY_JOINT_STANDARDIZED | 2009: fit K=8
LEGACY_JOINT_STANDARDIZED | 2010: fit K=8
LEGACY_JOINT_STANDARDIZED | 2011: fit K=8
LEGACY_JOINT_STANDARDIZED | 2012: fit K=8
LEGACY_JOINT_STANDARDIZED | 2013: fit K=8
LEGACY_JOINT_STANDARDIZED | 2014: fit K=8
LEGACY_JOINT_STANDARDIZED | 2015: fit K=8
LEGACY_JOINT_STANDARDIZED | 2016: fit K=8
LEGACY_JOINT_STANDARDIZED | 2017: fit K=8
LEGACY_JOINT_STANDARDIZED | 2018: fit K=8
LEGACY_JOINT_STANDARDIZED | 2019: fit K=8
LEGACY_JOINT_STANDARDIZED | 2020: fit K=8
LEGACY_JOINT_STANDARDIZED | 2021: fit K=8
LEGACY_JOINT_STANDARDIZED | 2022: fit K=8
LEGACY_JOINT_STANDARDIZED | 2023: fit K=8
LEGACY_JOINT_STANDARDIZED | 2024: fit K=8
LEGACY_JOINT_STANDARDIZED | 2025: fit K=8

PRICE_STANDARDIZED
PRICE_STANDARDIZED | 2008: fit K=8
PRICE_STANDARDIZED | 2009: fit K=8
PRICE_STANDARDIZED | 2010: fit K=8
PRICE_STANDARDIZED | 2011: fit K=8
PRICE_STANDARDIZED | 2012: fit K=8
PRICE_STANDARDIZED | 2

,spec_id,status,rows
0,ECONOMIC_COMPACT_WINSOR,OK,18
1,JOINT_ROBUST_PCA7,OK,18
2,LEGACY_JOINT_STANDARDIZED,OK,18
3,PRICE_ROBUST_PCA5,OK,18
4,PRICE_STANDARDIZED,OK,18
5,PRICE_WHITENED_PCA5,OK,18
6,VOL_ROBUST_PCA3,OK,18
7,VOL_STATIONARY,OK,18


### Runtime expectation

This is the expensive cell. It performs one fixed-K HMM fit per finalist per OOS year rather than a full K search in every fold. With eight direct finalists and 18 annual folds there are at most 144 HMM fits, mostly on lower-dimensional representations and capped at 75 EM iterations. Progress is printed continuously.

In [ ]:
# =========================================================
# 10) EXTRACT CANDIDATE-DATE FEATURES FOR EACH FINALIST
# =========================================================
def extract_candidate_features(spec, states, base_df):
    spec_id = spec["spec_id"]
    if states is None or len(states) == 0:
        return pd.DataFrame()
    grouped = {(int(y), str(m)): g.sort_values("week").reset_index(drop=True) for (y, m), g in states.groupby(["fold_year", "market"])}
    rows = []
    for _, r in base_df.iterrows():
        y = int(r["test_year"]); market = str(r["market"]); key = (y, market)
        if key not in grouped:
            continue
        g = grouped[key]; cand_date = pd.Timestamp(r["candidate_date"])
        eligible = g[g["week"] <= cand_date]
        if len(eligible) == 0:
            continue
        pos = int(eligible.index[-1]); cur = g.loc[pos]
        d = candidate_direction_sign(r["candidate_direction"])
        if not np.isfinite(d) or d == 0:
            continue
        d = 1.0 if d > 0 else -1.0
        out = {
            "candidate_date": cand_date, "market": market, "test_year": y,
            "candidate_direction": r["candidate_direction"],
            "category": r["category"] if "category" in r.index else None,
            "target": int(r["target"]),
            "book04_prob_genuine": float(r["book04_prob_genuine"]),
            "hmm_max_prob": float(cur["hmm_max_prob"]),
            "hmm_entropy": float(cur["hmm_entropy"]),
            "hmm_switch_probability": float(cur["hmm_switch_probability"]),
            "hmm_expected_persistence": float(cur["hmm_expected_persistence"]),
        }
        if "hmm_direction_score" in g.columns:
            out["hmm_candidate_dir_state_score"] = d * float(cur["hmm_direction_score"])
            if d > 0:
                successor_now = float(cur["hmm_prob_positive_state"]); incumbent_now = float(cur["hmm_prob_negative_state"])
            else:
                successor_now = float(cur["hmm_prob_negative_state"]); incumbent_now = float(cur["hmm_prob_positive_state"])
            out["hmm_successor_state_probability"] = successor_now
            out["hmm_incumbent_state_probability"] = incumbent_now
            out["hmm_successor_minus_incumbent"] = successor_now - incumbent_now
        if "hmm_volatility_score" in g.columns:
            out["hmm_volatility_score"] = float(cur["hmm_volatility_score"])
        for L in TOURNAMENT_LOOKBACKS:
            lag_pos = pos - L
            if lag_pos < 0:
                continue
            lag = g.loc[lag_pos]
            out[f"hmm_entropy_change_{L}w"] = float(cur["hmm_entropy"]) - float(lag["hmm_entropy"])
            out[f"hmm_switch_change_{L}w"] = float(cur["hmm_switch_probability"]) - float(lag["hmm_switch_probability"])
            if "hmm_direction_score" in g.columns:
                out[f"hmm_state_score_change_{L}w"] = d * (float(cur["hmm_direction_score"]) - float(lag["hmm_direction_score"]))
                successor_lag = float(lag["hmm_prob_positive_state"] if d > 0 else lag["hmm_prob_negative_state"])
                out[f"hmm_successor_prob_change_{L}w"] = successor_now - successor_lag
            if "hmm_volatility_score" in g.columns:
                out[f"hmm_volatility_score_change_{L}w"] = float(cur["hmm_volatility_score"]) - float(lag["hmm_volatility_score"])
        rows.append(out)
    out_df = pd.DataFrame(rows)
    if len(out_df):
        out_df["spec_id"] = spec_id
    return out_df

candidate_features_by_spec = {}
for spec in FINALIST_SPECS:
    sid = spec["spec_id"]
    cf = extract_candidate_features(spec, state_frames_by_spec[sid], base)
    candidate_features_by_spec[sid] = cf
    print(f"{sid}: {len(cf):,} candidates ({len(cf)/len(base):.1%} of frozen Book 04 events)")

direct_candidate_long = pd.concat([df for df in candidate_features_by_spec.values() if len(df)], ignore_index=True, sort=False)
direct_candidate_long.to_parquet(DATA_DIR / "v2_05_transformed_hmm_candidate_features.parquet", index=False)


LEGACY_JOINT_STANDARDIZED: 2,679 candidates (99.7% of frozen Book 04 events)
PRICE_STANDARDIZED: 2,679 candidates (99.7% of frozen Book 04 events)
PRICE_ROBUST_PCA5: 2,679 candidates (99.7% of frozen Book 04 events)
PRICE_WHITENED_PCA5: 2,679 candidates (99.7% of frozen Book 04 events)
VOL_STATIONARY: 2,679 candidates (99.7% of frozen Book 04 events)
VOL_ROBUST_PCA3: 2,679 candidates (99.7% of frozen Book 04 events)
JOINT_ROBUST_PCA7: 2,679 candidates (99.7% of frozen Book 04 events)
ECONOMIC_COMPACT_WINSOR: 2,679 candidates (99.7% of frozen Book 04 events)


In [ ]:
# =========================================================
# 11) BUILD PARALLEL PRICE-HMM + VOL-HMM CANDIDATE FEATURES
# =========================================================
price_sid = PARALLEL_SPEC["price_spec"]
vol_sid = PARALLEL_SPEC["vol_spec"]
price_cf = candidate_features_by_spec[price_sid].copy()
vol_cf = candidate_features_by_spec[vol_sid].copy()
key_cols = ["candidate_date", "market", "test_year", "candidate_direction", "category", "target", "book04_prob_genuine"]
price_feature_cols = [c for c in price_cf.columns if c.startswith("hmm_")]
vol_feature_cols = [c for c in vol_cf.columns if c.startswith("hmm_")]
p = price_cf[key_cols + price_feature_cols].rename(columns={c: f"price_{c}" for c in price_feature_cols})
v = vol_cf[key_cols + vol_feature_cols].rename(columns={c: f"vol_{c}" for c in vol_feature_cols})
parallel_cf = p.merge(v, on=key_cols, how="inner", validate="one_to_one")
parallel_cf["spec_id"] = PARALLEL_SPEC["spec_id"]
candidate_features_by_spec[PARALLEL_SPEC["spec_id"]] = parallel_cf
print(f"{PARALLEL_SPEC['spec_id']}: {len(parallel_cf):,} candidates ({len(parallel_cf)/len(base):.1%} of frozen Book 04 events)")
parallel_cf.to_parquet(DATA_DIR / "v2_05_parallel_hmm_candidate_features.parquet", index=False)


PARALLEL_PRICE_PCA5_VOL_PCA3: 2,679 candidates (99.7% of frozen Book 04 events)


## Candidate-level incremental test

For each specification and annual test year, frozen Book 04 is left untouched. One unweighted RF uses only the HMM features, and a second uses `book04_prob_genuine + HMM features`. Both train only on earlier OOS candidate years.

In [ ]:
# =========================================================
# 12) EXPANDING-WINDOW HMM-ONLY + BOOK04+HMM MODELS
# =========================================================
def make_incremental_rf():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("model", RandomForestClassifier(
            n_estimators=CONFIG["rf_estimators"],
            min_samples_leaf=CONFIG["rf_min_samples_leaf"],
            max_features=CONFIG["rf_max_features"],
            class_weight=None,
            n_jobs=-1,
            random_state=CONFIG["random_state"],
        )),
    ])

def hmm_feature_columns(df):
    return [c for c in df.columns if c.startswith("hmm_") or c.startswith("price_hmm_") or c.startswith("vol_hmm_")]

prediction_parts = []
for spec_id, cf in candidate_features_by_spec.items():
    if cf is None or len(cf) == 0:
        continue
    features = hmm_feature_columns(cf)
    print(f"\n{spec_id}: {len(features)} HMM features")
    for test_year in sorted(cf["test_year"].unique()):
        test = cf[cf["test_year"] == test_year].copy()
        train = cf[cf["test_year"] < test_year].copy()
        if len(train) < 100 or train["target"].nunique() < 2:
            continue
        identity = test[["candidate_date", "market", "category", "candidate_direction", "target", "test_year"]].copy()
        identity["spec_id"] = spec_id
        p0 = identity.copy(); p0["variant"] = "BOOK04"; p0["prob_genuine"] = test["book04_prob_genuine"].to_numpy(); prediction_parts.append(p0)
        mh = make_incremental_rf(); mh.fit(train[features], train["target"])
        p1 = identity.copy(); p1["variant"] = "HMM_ONLY"; p1["prob_genuine"] = mh.predict_proba(test[features])[:,1]; prediction_parts.append(p1)
        stack_features = ["book04_prob_genuine"] + features
        ms = make_incremental_rf(); ms.fit(train[stack_features], train["target"])
        p2 = identity.copy(); p2["variant"] = "BOOK04_PLUS_HMM"; p2["prob_genuine"] = ms.predict_proba(test[stack_features])[:,1]; prediction_parts.append(p2)

tournament_predictions = pd.concat(prediction_parts, ignore_index=True)
display(tournament_predictions.groupby(["spec_id", "variant"]).size().rename("rows").reset_index())



LEGACY_JOINT_STANDARDIZED: 24 HMM features

PRICE_STANDARDIZED: 20 HMM features

PRICE_ROBUST_PCA5: 20 HMM features

PRICE_WHITENED_PCA5: 20 HMM features

VOL_STATIONARY: 14 HMM features

VOL_ROBUST_PCA3: 14 HMM features

JOINT_ROBUST_PCA7: 24 HMM features

ECONOMIC_COMPACT_WINSOR: 24 HMM features

PARALLEL_PRICE_PCA5_VOL_PCA3: 34 HMM features


,spec_id,variant,rows
0,ECONOMIC_COMPACT_WINSOR,BOOK04,2561
1,ECONOMIC_COMPACT_WINSOR,BOOK04_PLUS_HMM,2561
2,ECONOMIC_COMPACT_WINSOR,HMM_ONLY,2561
3,JOINT_ROBUST_PCA7,BOOK04,2561
4,JOINT_ROBUST_PCA7,BOOK04_PLUS_HMM,2561
5,JOINT_ROBUST_PCA7,HMM_ONLY,2561
6,LEGACY_JOINT_STANDARDIZED,BOOK04,2561
7,LEGACY_JOINT_STANDARDIZED,BOOK04_PLUS_HMM,2561
8,LEGACY_JOINT_STANDARDIZED,HMM_ONLY,2561
9,PARALLEL_PRICE_PCA5_VOL_PCA3,BOOK04,2561


In [ ]:
# =========================================================
# 13) FAIR PAIRWISE METRICS + DELTAS VERSUS FROZEN BOOK 04
# =========================================================
def safe_auc(y, p):
    return roc_auc_score(y, p) if pd.Series(y).nunique() > 1 else np.nan

def safe_pr(y, p):
    return average_precision_score(y, p) if pd.Series(y).nunique() > 1 else np.nan

def metric_row(g):
    y = g["target"].astype(int).to_numpy()
    p = np.clip(g["prob_genuine"].astype(float).to_numpy(), 1e-6, 1-1e-6)
    base_rate = float(np.mean(y))
    temp = g.copy()
    temp["rank_pct"] = temp.groupby("test_year")["prob_genuine"].rank(method="average", pct=True)
    top = temp[temp["rank_pct"] >= 1 - CONFIG["top_quantile"]]
    top_rate = top["target"].mean() if len(top) else np.nan
    return pd.Series({
        "events": len(g), "genuine_rate": base_rate,
        "roc_auc": safe_auc(y,p), "pr_auc": safe_pr(y,p),
        "brier": brier_score_loss(y,p), "log_loss": log_loss(y,p,labels=[0,1]),
        "mean_predicted_probability": float(np.mean(p)),
        "top_quintile_events": len(top), "top_quintile_genuine_rate": top_rate,
        "top_quintile_lift": top_rate/base_rate if base_rate > 0 else np.nan,
    })

pairwise_metrics = tournament_predictions.groupby(["spec_id", "variant"], group_keys=False).apply(metric_row).reset_index()
b4m = pairwise_metrics[pairwise_metrics["variant"]=="BOOK04"].set_index("spec_id")
stkm = pairwise_metrics[pairwise_metrics["variant"]=="BOOK04_PLUS_HMM"].set_index("spec_id")
delta_rows = []
for sid in sorted(set(b4m.index).intersection(stkm.index)):
    delta_rows.append({
        "spec_id": sid,
        "delta_roc_auc_vs_book04": stkm.loc[sid,"roc_auc"]-b4m.loc[sid,"roc_auc"],
        "delta_pr_auc_vs_book04": stkm.loc[sid,"pr_auc"]-b4m.loc[sid,"pr_auc"],
        "delta_brier_vs_book04": stkm.loc[sid,"brier"]-b4m.loc[sid,"brier"],
        "delta_log_loss_vs_book04": stkm.loc[sid,"log_loss"]-b4m.loc[sid,"log_loss"],
        "delta_top_quintile_rate_vs_book04": stkm.loc[sid,"top_quintile_genuine_rate"]-b4m.loc[sid,"top_quintile_genuine_rate"],
        "delta_top_quintile_lift_vs_book04": stkm.loc[sid,"top_quintile_lift"]-b4m.loc[sid,"top_quintile_lift"],
    })
incremental_deltas = pd.DataFrame(delta_rows).sort_values("delta_roc_auc_vs_book04", ascending=False)
print("PAIRWISE METRICS"); display(pairwise_metrics)
print("INCREMENTAL DELTAS VS EXACT FROZEN BOOK 04"); display(incremental_deltas)


PAIRWISE METRICS


,spec_id,variant,events,genuine_rate,roc_auc,pr_auc,brier,log_loss,mean_predicted_probability,top_quintile_events,top_quintile_genuine_rate,top_quintile_lift
0,ECONOMIC_COMPACT_WINSOR,BOOK04,2561.0,0.323311,0.753542,0.580105,0.182912,0.545683,0.324698,522.0,0.643678,1.990893
1,ECONOMIC_COMPACT_WINSOR,BOOK04_PLUS_HMM,2561.0,0.323311,0.703422,0.496088,0.195800,0.575960,0.344347,522.0,0.555556,1.718331
2,ECONOMIC_COMPACT_WINSOR,HMM_ONLY,2561.0,0.323311,0.575443,0.375343,0.218440,0.629484,0.336061,522.0,0.411877,1.273935
3,JOINT_ROBUST_PCA7,BOOK04,2561.0,0.323311,0.753542,0.580105,0.182912,0.545683,0.324698,522.0,0.643678,1.990893
4,JOINT_ROBUST_PCA7,BOOK04_PLUS_HMM,2561.0,0.323311,0.692467,0.504535,0.197605,0.581173,0.342635,522.0,0.561303,1.736106
5,JOINT_ROBUST_PCA7,HMM_ONLY,2561.0,0.323311,0.501311,0.317928,0.225449,0.644167,0.333749,522.0,0.308429,0.953970
6,LEGACY_JOINT_STANDARDIZED,BOOK04,2561.0,0.323311,0.753542,0.580105,0.182912,0.545683,0.324698,522.0,0.643678,1.990893
7,LEGACY_JOINT_STANDARDIZED,BOOK04_PLUS_HMM,2561.0,0.323311,0.706795,0.525438,0.194408,0.572816,0.350893,522.0,0.567050,1.753882
8,LEGACY_JOINT_STANDARDIZED,HMM_ONLY,2561.0,0.323311,0.593558,0.408295,0.214886,0.619990,0.343966,522.0,0.417625,1.291711
9,PARALLEL_PRICE_PCA5_VOL_PCA3,BOOK04,2561.0,0.323311,0.753542,0.580105,0.182912,0.545683,0.324698,522.0,0.643678,1.990893


INCREMENTAL DELTAS VS EXACT FROZEN BOOK 04


,spec_id,delta_roc_auc_vs_book04,delta_pr_auc_vs_book04,delta_brier_vs_book04,delta_log_loss_vs_book04,delta_top_quintile_rate_vs_book04,delta_top_quintile_lift_vs_book04
2,LEGACY_JOINT_STANDARDIZED,-0.046746,-0.054667,0.011497,0.027133,-0.076628,-0.237011
0,ECONOMIC_COMPACT_WINSOR,-0.050119,-0.084017,0.012888,0.030277,-0.088123,-0.272563
6,PRICE_WHITENED_PCA5,-0.050402,-0.084146,0.012879,0.031094,-0.088123,-0.272563
8,VOL_STATIONARY,-0.052864,-0.084489,0.013059,0.032262,-0.070881,-0.219235
5,PRICE_STANDARDIZED,-0.054596,-0.087632,0.013611,0.031615,-0.090038,-0.278488
7,VOL_ROBUST_PCA3,-0.057506,-0.082972,0.014035,0.035103,-0.059387,-0.183684
1,JOINT_ROBUST_PCA7,-0.061074,-0.075570,0.014693,0.035490,-0.082375,-0.254787
4,PRICE_ROBUST_PCA5,-0.062828,-0.071686,0.015020,0.035989,-0.090038,-0.278488
3,PARALLEL_PRICE_PCA5_VOL_PCA3,-0.074633,-0.090286,0.018015,0.043576,-0.072797,-0.225161


In [ ]:
# =========================================================
# 14) DIRECTION / ASSET-CLASS / YEARLY ROBUSTNESS
# =========================================================
tournament_predictions["direction_name"] = np.where(
    tournament_predictions["candidate_direction"].apply(candidate_direction_sign) > 0,
    "BEAR_TO_BULL", "BULL_TO_BEAR"
)
metrics_by_direction = tournament_predictions.groupby(["spec_id","variant","direction_name"], group_keys=False).apply(metric_row).reset_index()
metrics_by_asset_class = tournament_predictions.groupby(["spec_id","variant","category"], group_keys=False).apply(metric_row).reset_index()
yearly_metrics = tournament_predictions.groupby(["spec_id","variant","test_year"], group_keys=False).apply(metric_row).reset_index()
yearly_stability = yearly_metrics.groupby(["spec_id","variant"]).agg(
    test_years=("test_year","nunique"),
    mean_yearly_auc=("roc_auc","mean"), median_yearly_auc=("roc_auc","median"),
    pct_years_auc_above_050=("roc_auc", lambda s: np.mean(pd.Series(s).dropna()>0.50)),
    mean_yearly_pr_auc=("pr_auc","mean"), mean_yearly_brier=("brier","mean"),
    mean_yearly_log_loss=("log_loss","mean"),
).reset_index()
display(yearly_stability[yearly_stability["variant"]=="BOOK04_PLUS_HMM"].sort_values("mean_yearly_auc", ascending=False))


,spec_id,variant,test_years,mean_yearly_auc,median_yearly_auc,pct_years_auc_above_050,mean_yearly_pr_auc,mean_yearly_brier,mean_yearly_log_loss
7,LEGACY_JOINT_STANDARDIZED,BOOK04_PLUS_HMM,17,0.719871,0.710559,1.0,0.538819,0.193759,0.571328
1,ECONOMIC_COMPACT_WINSOR,BOOK04_PLUS_HMM,17,0.719634,0.713250,1.0,0.535425,0.194308,0.572600
16,PRICE_STANDARDIZED,BOOK04_PLUS_HMM,17,0.715357,0.695068,1.0,0.533043,0.195498,0.574770
22,VOL_ROBUST_PCA3,BOOK04_PLUS_HMM,17,0.711598,0.699248,1.0,0.542391,0.196233,0.579261
19,PRICE_WHITENED_PCA5,BOOK04_PLUS_HMM,17,0.710287,0.728483,1.0,0.526374,0.195015,0.575102
25,VOL_STATIONARY,BOOK04_PLUS_HMM,17,0.706989,0.693206,1.0,0.534886,0.195252,0.576379
4,JOINT_ROBUST_PCA7,BOOK04_PLUS_HMM,17,0.703243,0.707115,1.0,0.540430,0.196821,0.579475
13,PRICE_ROBUST_PCA5,BOOK04_PLUS_HMM,17,0.702994,0.671895,1.0,0.539828,0.197123,0.579861
10,PARALLEL_PRICE_PCA5_VOL_PCA3,BOOK04_PLUS_HMM,17,0.692760,0.673913,1.0,0.531944,0.200067,0.587419


In [ ]:
# =========================================================
# 15) LATENT-STATE DESCRIPTIVE AUDIT BY SPECIFICATION
# =========================================================
audit_rows = []
for sid, cf in candidate_features_by_spec.items():
    if cf is None or len(cf)==0:
        continue
    for target, g in cf.groupby("target"):
        row = {"spec_id": sid, "target": int(target), "label_name": "GENUINE" if int(target)==1 else "FAILED", "events": len(g)}
        for c in [
            "hmm_successor_state_probability","hmm_incumbent_state_probability","hmm_successor_minus_incumbent",
            "hmm_candidate_dir_state_score","hmm_volatility_score","hmm_successor_prob_change_4w","hmm_successor_prob_change_8w",
            "hmm_state_score_change_4w","hmm_state_score_change_8w","hmm_volatility_score_change_4w","hmm_volatility_score_change_8w",
            "price_hmm_successor_state_probability","price_hmm_successor_prob_change_8w","vol_hmm_volatility_score","vol_hmm_volatility_score_change_8w",
        ]:
            if c in g.columns:
                row[f"mean_{c}"] = g[c].mean()
        audit_rows.append(row)
candidate_state_audit = pd.DataFrame(audit_rows)
display(candidate_state_audit)


,spec_id,target,label_name,events,mean_hmm_successor_state_probability,mean_hmm_incumbent_state_probability,mean_hmm_successor_minus_incumbent,mean_hmm_candidate_dir_state_score,mean_hmm_volatility_score,mean_hmm_successor_prob_change_4w,mean_hmm_successor_prob_change_8w,mean_hmm_state_score_change_4w,mean_hmm_state_score_change_8w,mean_hmm_volatility_score_change_4w,mean_hmm_volatility_score_change_8w,mean_price_hmm_successor_state_probability,mean_price_hmm_successor_prob_change_8w,mean_vol_hmm_volatility_score,mean_vol_hmm_volatility_score_change_8w
0,LEGACY_JOINT_STANDARDIZED,0,FAILED,1807,0.431685,0.568315,-0.136630,-0.191179,-0.090764,0.081033,0.125530,0.136309,0.228404,-0.031418,-0.050745,NaN,NaN,NaN,NaN
1,LEGACY_JOINT_STANDARDIZED,1,GENUINE,872,0.577610,0.422390,0.155221,0.038473,-0.198704,0.107390,0.161361,0.153842,0.236815,-0.036385,-0.054291,NaN,NaN,NaN,NaN
2,PRICE_STANDARDIZED,0,FAILED,1807,0.457504,0.542496,-0.084993,-0.181725,NaN,0.091105,0.137024,0.124051,0.214187,NaN,NaN,NaN,NaN,NaN,NaN
3,PRICE_STANDARDIZED,1,GENUINE,872,0.609758,0.390242,0.219516,0.017576,NaN,0.130158,0.181119,0.148120,0.238874,NaN,NaN,NaN,NaN,NaN,NaN
4,PRICE_ROBUST_PCA5,0,FAILED,1807,0.402905,0.597095,-0.194190,-0.198388,NaN,0.023859,0.043690,0.076431,0.136458,NaN,NaN,NaN,NaN,NaN,NaN
5,PRICE_ROBUST_PCA5,1,GENUINE,872,0.453082,0.546918,-0.093835,-0.032302,NaN,0.047851,0.088832,0.114258,0.205602,NaN,NaN,NaN,NaN,NaN,NaN
6,PRICE_WHITENED_PCA5,0,FAILED,1807,0.365239,0.634761,-0.269521,-0.171683,NaN,0.015986,0.040787,0.079033,0.151121,NaN,NaN,NaN,NaN,NaN,NaN
7,PRICE_WHITENED_PCA5,1,GENUINE,872,0.362337,0.637663,-0.275325,-0.087555,NaN,0.034298,0.066813,0.066789,0.124114,NaN,NaN,NaN,NaN,NaN,NaN
8,VOL_STATIONARY,0,FAILED,1807,NaN,NaN,NaN,NaN,0.433927,NaN,NaN,NaN,NaN,0.004887,0.004566,NaN,NaN,NaN,NaN
9,VOL_STATIONARY,1,GENUINE,872,NaN,NaN,NaN,NaN,0.412676,NaN,NaN,NaN,NaN,0.006145,0.002895,NaN,NaN,NaN,NaN


In [ ]:
# =========================================================
# 16) EXACT FROZEN BOOK 04 AUDIT ON EACH SPECIFICATION SAMPLE
# =========================================================
frozen_audit = pairwise_metrics[pairwise_metrics["variant"]=="BOOK04"].copy().sort_values("spec_id")
frozen_audit["note"] = "Exact saved Book 04 OOS probabilities on the same events/years available to each transformed-HMM specification."
display(frozen_audit)


,spec_id,variant,events,genuine_rate,roc_auc,pr_auc,brier,log_loss,mean_predicted_probability,top_quintile_events,top_quintile_genuine_rate,top_quintile_lift,note
0,ECONOMIC_COMPACT_WINSOR,BOOK04,2561.0,0.323311,0.753542,0.580105,0.182912,0.545683,0.324698,522.0,0.643678,1.990893,Exact saved Book 04 OOS probabilities on the s...
3,JOINT_ROBUST_PCA7,BOOK04,2561.0,0.323311,0.753542,0.580105,0.182912,0.545683,0.324698,522.0,0.643678,1.990893,Exact saved Book 04 OOS probabilities on the s...
6,LEGACY_JOINT_STANDARDIZED,BOOK04,2561.0,0.323311,0.753542,0.580105,0.182912,0.545683,0.324698,522.0,0.643678,1.990893,Exact saved Book 04 OOS probabilities on the s...
9,PARALLEL_PRICE_PCA5_VOL_PCA3,BOOK04,2561.0,0.323311,0.753542,0.580105,0.182912,0.545683,0.324698,522.0,0.643678,1.990893,Exact saved Book 04 OOS probabilities on the s...
12,PRICE_ROBUST_PCA5,BOOK04,2561.0,0.323311,0.753542,0.580105,0.182912,0.545683,0.324698,522.0,0.643678,1.990893,Exact saved Book 04 OOS probabilities on the s...
15,PRICE_STANDARDIZED,BOOK04,2561.0,0.323311,0.753542,0.580105,0.182912,0.545683,0.324698,522.0,0.643678,1.990893,Exact saved Book 04 OOS probabilities on the s...
18,PRICE_WHITENED_PCA5,BOOK04,2561.0,0.323311,0.753542,0.580105,0.182912,0.545683,0.324698,522.0,0.643678,1.990893,Exact saved Book 04 OOS probabilities on the s...
21,VOL_ROBUST_PCA3,BOOK04,2561.0,0.323311,0.753542,0.580105,0.182912,0.545683,0.324698,522.0,0.643678,1.990893,Exact saved Book 04 OOS probabilities on the s...
24,VOL_STATIONARY,BOOK04,2561.0,0.323311,0.753542,0.580105,0.182912,0.545683,0.324698,522.0,0.643678,1.990893,Exact saved Book 04 OOS probabilities on the s...


## Optional wider-K robustness

The tournament fixes `K=8` across specifications so preprocessing and information-set comparisons are not confounded by different state-count searches. If a transformed model materially improves the frozen Book 04 combination, set `CONFIG["run_extended_k_robustness"] = True` and rerun the optional cell below. It evaluates `K=3..10` only for the strongest transformed specification(s).

In [ ]:
# =========================================================
# 17) OPTIONAL K=3..10 ROBUSTNESS FOR THE BEST TRANSFORMED SPEC
# =========================================================
k_robustness = pd.DataFrame()
if CONFIG["run_extended_k_robustness"]:
    if len(incremental_deltas)==0:
        raise RuntimeError("No transformed-HMM incremental results are available.")
    best_sid = incremental_deltas.iloc[0]["spec_id"]
    robustness_sids = [price_sid, vol_sid] if best_sid == PARALLEL_SPEC["spec_id"] else [best_sid]
    reference_cutoff = pd.Timestamp(f"{CONFIG['first_test_year']}-01-01")
    reference_train = weekly[weekly["week"] < reference_cutoff].copy()
    spec_lookup = {s["spec_id"]: s for s in FINALIST_SPECS}
    rows = []
    for sid in robustness_sids:
        spec = spec_lookup[sid]
        for k in range(3,11):
            try:
                r = chronological_unsupervised_score(reference_train, spec["architecture"], spec["preprocessing"], k, n_components=spec["n_components"])
                r["spec_id"] = sid; rows.append(r)
            except Exception as e:
                rows.append({"spec_id": sid, "n_states": k, "error": str(e)})
    k_robustness = pd.DataFrame(rows); display(k_robustness)
else:
    print("Extended K robustness skipped (default).")


Extended K robustness skipped (default).


In [ ]:
# =========================================================
# 18) SAVE DEFINITIVE BOOK 05 OUTPUTS
# =========================================================
pairwise_metrics.to_csv(RESULTS_DIR / "v2_05_transformed_overall_metrics.csv", index=False)
incremental_deltas.to_csv(RESULTS_DIR / "v2_05_transformed_incremental_deltas.csv", index=False)
metrics_by_direction.to_csv(RESULTS_DIR / "v2_05_transformed_metrics_by_direction.csv", index=False)
metrics_by_asset_class.to_csv(RESULTS_DIR / "v2_05_transformed_metrics_by_asset_class.csv", index=False)
yearly_metrics.to_csv(RESULTS_DIR / "v2_05_transformed_yearly_metrics.csv", index=False)
yearly_stability.to_csv(RESULTS_DIR / "v2_05_transformed_yearly_stability.csv", index=False)
candidate_state_audit.to_csv(RESULTS_DIR / "v2_05_transformed_candidate_state_audit.csv", index=False)
frozen_audit.to_csv(RESULTS_DIR / "v2_05_transformed_frozen_book04_audit.csv", index=False)
tournament_predictions.to_parquet(DATA_DIR / "v2_05_transformed_oos_predictions.parquet", index=False)
if len(k_robustness):
    k_robustness.to_csv(RESULTS_DIR / "v2_05_transformed_k_robustness.csv", index=False)
config_save = CONFIG.copy()
config_save["finalist_specs"] = FINALIST_SPECS
config_save["parallel_spec"] = PARALLEL_SPEC
config_save["tournament_lookbacks"] = TOURNAMENT_LOOKBACKS
config_save["frozen_book04_model"] = FROZEN_MODEL
with open(CONFIG_DIR / "v2_05_config.json", "w") as f:
    json.dump(config_save, f, indent=2, default=str)
print("Definitive Book 05 outputs saved.")


Definitive Book 05 outputs saved.


# Outputs to upload for analysis

### Required
1. `v2_05_transformed_overall_metrics.csv`
2. `v2_05_transformed_incremental_deltas.csv`
3. `v2_05_transformed_yearly_stability.csv`
4. `v2_05_transformed_metrics_by_direction.csv`
5. `v2_05_transformed_candidate_state_audit.csv`
6. `v2_05_transformed_frozen_book04_audit.csv`
7. `v2_05_transformed_hmm_fit_audit.csv`

### Preferred
8. `v2_05_transformed_metrics_by_asset_class.csv`
9. `v2_05_transformed_yearly_metrics.csv`
10. `v2_05_preprocessing_screen.csv`
11. `v2_05_stage_b_plan.csv`

If extended K robustness is enabled, also upload `v2_05_transformed_k_robustness.csv`.

The completion gate is the `BOOK04_PLUS_HMM` delta table. A transformed HMM survives only if it improves the frozen Book 04 model OOS rather than merely improving standalone HMM fit.

## Research Outcome

Book 05 investigated whether latent market-state modelling can identify transition dynamics that are not already captured by conventional transition geometry and SuperbCommand.

The original hypothesis was economically intuitive: genuine trend transitions may involve a gradual migration of probability mass from an incumbent latent regime toward a prospective successor regime before conventional trend confirmation. Hidden Markov Models therefore offered a natural framework for modelling unobserved market states and state-transition probabilities.

The hypothesis was subjected to progressively stronger tests.

Initial candidate-sequence HMMs were rejected methodologically because sparse transition candidates do not constitute a continuous market-time sequence and conventional smoothed HMM posteriors can introduce future information. Book 05 was therefore rebuilt around continuous weekly market histories, separate market sequences and explicitly causal forward filtering.

The research was then extended to test whether the HMM's performance was being constrained by its input representation. The final programme examined:

- raw and stationary factor representations;
- standardisation;
- winsorisation;
- nonlinear transformations;
- PCA dimensionality reduction;
- robust PCA;
- whitened PCA;
- price/trend-only HMMs;
- volatility-only HMMs;
- joint price-and-volatility HMMs;
- deliberately compact economic-state HMMs;
- parallel price-HMM and volatility-HMM processes.

These tests confirm that the original feature space contains substantial redundancy and that latent-state migration is empirically observable. For example, genuine transitions generally exhibit greater probability mass in successor-direction states and stronger migration toward those states than failed transitions.

However, **latent-state information does not provide incremental predictive alpha beyond the frozen Book 04 model**.

On the common transformed-HMM evaluation sample, frozen Book 04 retains approximately:

- **ROC AUC: 0.754**;
- **PR AUC: 0.580**;
- **Brier score: 0.183**;
- top-quintile genuine-transition rate: **64.4%**.

No tested HMM architecture improves these results. Even the strongest HMM augmentation reduces ROC AUC to approximately **0.707**, while other transformed specifications deteriorate performance further. Every HMM augmentation also worsens probability scoring and reduces top-quintile transition enrichment.

The negative result cannot readily be attributed to numerical instability: the definitive transformed-HMM tournament completed **144 direct finalist fits successfully**. Nor is it explained simply by correlated raw factors, inappropriate scaling, excessive dimensionality, mixing volatility with direction, or insufficiently flexible latent-state representations.

### Conclusion

Book 05 supports an important distinction:

> **Latent regime migration exists, but HMMs do not convert that structure into incremental predictive information beyond the simpler transition model.**

The research therefore rejects HMMs from the production V2 architecture. This rejection survives continuous market-time modelling, causal filtering, alternative preprocessing, dimensionality reduction and economically distinct price/volatility state architectures.

The result strengthens the parsimonious Book 04 specification:

\[
\boxed{
\text{Conventional Transition Geometry}
+
\text{SuperbCommand State}
}
\]

More sophisticated latent-state machinery is not retained merely because it produces economically interpretable regimes.

**Status: FROZEN. HMM layer REJECTED from the production V2 transition model.**